# ⭐ Bài tập về nhà 3 — Export YOLO26n sang ONNX, đo latency trên CPU
Lab Ngày 18 · Track 4. Đề: *Export `yolo26n.pt` sang ONNX (`model.export(format="onnx")`), đo latency trên CPU của head
one-to-many + NMS và head one-to-one ở conf 0.001 và 0.25.*

- Chạy trên Colab **runtime CPU** (không GPU), cùng `ultralytics==8.4.171` như notebook chính.
- Cách đo giống ô `bench` ở mục 1C: warm-up 1 lần, rồi lấy trung bình `Results.speed` của 30 lần chạy trên `bus.jpg`.
- Đo thêm bản PyTorch trên cùng CPU để có mốc so sánh với ONNX Runtime.

In [ ]:
%pip install -q "ultralytics==8.4.171" onnx onnxslim onnxruntime

In [ ]:
import os, platform, shutil
from pathlib import Path

import onnxruntime as ort
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

IMG_BUS = str(Path(ultralytics.__file__).parent / "assets" / "bus.jpg")
cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
print(f"torch {torch.__version__} | ultralytics {ultralytics.__version__} | onnxruntime {ort.__version__}")
print(f"CPU: {cpu} · {os.cpu_count()} luồng · torch.get_num_threads() = {torch.get_num_threads()} · CUDA: {torch.cuda.is_available()}")

## 1. Export hai head sang ONNX
Ở `ultralytics 8.4.171`, tham số `nms` của `export` chọn head:
- `nms=None` (mặc định của `model.export(format="onnx")`): giữ head **one-to-many**, output thô `(1, 84, 8400)`, NMS chạy **ngoài** graph khi suy luận;
- `nms=False`: chọn head **one-to-one**, NMS-free, output `(1, 300, 6)` đã là danh sách box cuối cùng.

In [ ]:
ONNX = {}
for head, kw in {"one-to-many": {}, "one-to-one": {"nms": False}}.items():
    f = YOLO("yolo26n.pt").export(format="onnx", imgsz=640, device="cpu", **kw)
    ONNX[head] = shutil.move(f, f"yolo26n_{head}.onnx")

for head, path in ONNX.items():
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    print(f"{head:12s} {path}: input {sess.get_inputs()[0].shape} → output {sess.get_outputs()[0].shape} · "
          f"{Path(path).stat().st_size / 1e6:.1f} MB · provider {sess.get_providers()[0]}")

## 2. Đo latency trên CPU
Bốn cấu hình của đề (2 head × 2 mức conf) cho ONNX Runtime, cộng bốn cấu hình tương ứng của PyTorch.
Dùng **object riêng cho mỗi head**, như notebook chính: một object PyTorch đã gọi với `nms=False` sẽ giữ head one-to-one.
Model ONNX có input cố định 640×640 nên `bus.jpg` được letterbox thành 640×640; bản PyTorch letterbox thành 640×480.

In [ ]:
def bench(model, n=30, **kw):
    """Thời gian trung bình (ms) của từng giai đoạn trên bus.jpg, lấy từ Results.speed."""
    model.predict(IMG_BUS, device="cpu", verbose=False, **kw)                 # warm-up
    acc = {"preprocess": 0.0, "inference": 0.0, "postprocess": 0.0}
    for _ in range(n):
        r = model.predict(IMG_BUS, device="cpu", verbose=False, **kw)[0]
        for k in acc:
            acc[k] += r.speed[k] / n
    return {f"{k} (ms)": round(v, 2) for k, v in acc.items()} | {"tổng (ms)": round(sum(acc.values()), 2), "số box": len(r.boxes)}


CONFIGS = {
    ("ONNX", "one-to-many + NMS"): (YOLO(ONNX["one-to-many"], task="detect"), {"iou": 0.7}),
    ("ONNX", "one-to-one NMS-free"): (YOLO(ONNX["one-to-one"], task="detect"), {}),
    ("PyTorch", "one-to-many + NMS"): (YOLO("yolo26n.pt"), {"iou": 0.7}),
    ("PyTorch", "one-to-one NMS-free"): (YOLO("yolo26n.pt"), {"nms": False}),
}
LATENCY_CPU = [{"Backend": backend, "Head": head, "conf": conf} | bench(model, conf=conf, **kw)
               for (backend, head), (model, kw) in CONFIGS.items() for conf in (0.25, 0.001)]
df = pd.DataFrame(LATENCY_CPU)
print(f"Latency YOLO26n trên CPU ({cpu}), trung bình 30 lần:")
df

## 3. Riêng phần hậu xử lý: NMS tốn bao nhiêu trên CPU?

In [ ]:
key = ["Backend", "conf"]
o2m = df[df.Head.str.startswith("one-to-many")].set_index(key)
o2o = df[df.Head.str.startswith("one-to-one")].set_index(key)
cmp = pd.DataFrame({
    "box sau NMS (one-to-many)": o2m["số box"],
    "postprocess one-to-many + NMS (ms)": o2m["postprocess (ms)"],
    "postprocess one-to-one (ms)": o2o["postprocess (ms)"],
    "postprocess chậm hơn (lần)": (o2m["postprocess (ms)"] / o2o["postprocess (ms)"]).round(1),
    "tổng one-to-many − one-to-one (ms)": (o2m["tổng (ms)"] - o2o["tổng (ms)"]).round(2),
})
cmp